In [ ]:
# Install required package
!pip install nba_api

import json
from nba_api.stats.endpoints import teamestimatedmetrics
from nba_api.stats.endpoints import TeamVsPlayer
from nba_api.stats.endpoints import PlayerGameLogs
from nba_api.stats.endpoints import playergamelog
import pandas as pd
import requests
from datetime import datetime
import pytz
from nba_api.stats.static import teams
from nba_api.stats.endpoints import commonteamroster
from nba_api.stats.library.parameters import SeasonAll
from IPython.display import display
import time  # Ensure time is imported

# Set display options for Google Colab
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

# Fetch today's schedule from NBA Stats API
def fetch_schedule():
    try:
        url = "https://data.nba.com/data/10s/v2015/json/mobile_teams/nba/2024/league/00_full_schedule.json"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Set timezone to EST
        est = pytz.timezone('US/Eastern')
        today = datetime.now(est).strftime('%Y-%m-%d')
        games_today = []

        for month_data in data['lscd']:
            for game in month_data['mscd']['g']:
                if game['gdte'] == today:
                    games_today.append({
                        'home_team': game['h']['ta'],
                        'away_team': game['v']['ta'],
                        'home_team_id': game['h']['tid'],
                        'away_team_id': game['v']['tid']
                    })

        if not games_today:
            print(f"No games scheduled for {today}.")
        return games_today
    except Exception as e:
        print(f"Error fetching schedule: {e}")
        return []

# Function to fetch DvP stats
def fetch_dvp_stats(team_id, season_type="Regular Season"):
    try:
        stats = TeamVsPlayer(team_id=team_id, season_type=season_type)
        return stats.get_data_frames()[0]  # return the first dataframe
    except Exception as e:
        print(f"Error fetching DvP stats for team {team_id}: {e}")
        return None

# Function to process player data
def process_players(team):
    team_players = team['players']
    player_stats = []
    for player in team_players:
        player_data = {
            'Player': player['full_name'],
            'Position': player.get('position', 'Unknown')  # Handle missing 'POS' field
        }
        player_stats.append(player_data)
    return player_stats

# Fetch team metrics
def fetch_team_metrics():
    try:
        team_metrics = teamestimatedmetrics.TeamEstimatedMetrics(
            season="2024-25", season_type="Regular Season", league_id="00"
        )
        data = team_metrics.get_data_frames()[0]
        metrics = data[['TEAM_NAME', 'E_OFF_RATING', 'E_DEF_RATING', 'E_PACE']]
        metrics_dict = metrics.set_index('TEAM_NAME').to_dict(orient='index')

        # Calculate league averages
        avg_pace = metrics['E_PACE'].mean()
        avg_off_rating = metrics['E_OFF_RATING'].mean()
        avg_def_rating = metrics['E_DEF_RATING'].mean()

        # Map abbreviations to full names
        team_name_mapping = {
            "ATL": "Atlanta Hawks", "BOS": "Boston Celtics", "BKN": "Brooklyn Nets", "CHA": "Charlotte Hornets",
            "CHI": "Chicago Bulls", "CLE": "Cleveland Cavaliers", "DAL": "Dallas Mavericks", "DEN": "Denver Nuggets",
            "DET": "Detroit Pistons", "GSW": "Golden State Warriors", "HOU": "Houston Rockets", "IND": "Indiana Pacers",
            "LAC": "LA Clippers", "LAL": "Los Angeles Lakers", "MEM": "Memphis Grizzlies", "MIA": "Miami Heat",
            "MIL": "Milwaukee Bucks", "MIN": "Minnesota Timberwolves", "NOP": "New Orleans Pelicans", "NYK": "New York Knicks",
            "OKC": "Oklahoma City Thunder", "ORL": "Orlando Magic", "PHI": "Philadelphia 76ers", "PHX": "Phoenix Suns",
            "POR": "Portland Trail Blazers", "SAC": "Sacramento Kings", "SAS": "San Antonio Spurs", "TOR": "Toronto Raptors",
            "UTA": "Utah Jazz", "WAS": "Washington Wizards"
        }

        # Adjusted metrics dictionary
        adjusted_metrics = {}
        for abbr, full_name in team_name_mapping.items():
            if full_name in metrics_dict:
                adjusted_metrics[abbr] = metrics_dict[full_name]
            else:
                print(f"Warning: Metrics not found for {abbr} ({full_name}).")

        return adjusted_metrics, avg_pace, avg_off_rating, avg_def_rating
    except Exception as e:
        print(f"Error fetching team metrics: {e}")
        return {}, 0, 0, 0

# Define the adjust_prediction function
def adjust_prediction(stat_mean, team_metrics, opposing_metrics, avg_pace, avg_off_rating, avg_def_rating):
    try:
        # Calculate pace adjustment factor
        pace_factor = team_metrics.get('E_PACE', avg_pace) / avg_pace if avg_pace != 0 else 1

        # Offensive adjustment factor
        offensive_factor = team_metrics.get('E_OFF_RATING', avg_off_rating) / avg_off_rating if avg_off_rating != 0 else 1

        # Defensive adjustment factor (using opponent's defensive rating)
        defensive_factor = opposing_metrics.get('E_DEF_RATING', avg_def_rating) / avg_def_rating if avg_def_rating != 0 else 1

        # Apply adjustments to the stat mean
        adjusted_stat = stat_mean * pace_factor * offensive_factor / defensive_factor
        return adjusted_stat
    except Exception as e:
        print(f"Error adjusting prediction: {e}")
        return stat_mean

# Analyze teams and players for today's games
def analyze_teams_for_today():
    games_today = fetch_schedule()
    team_metrics, avg_pace, avg_off_rating, avg_def_rating = fetch_team_metrics()

    if not games_today:
        print("No games scheduled for today.")
        return pd.DataFrame()

    all_players_data = []

    for game in games_today:
        home_team = game['home_team']
        away_team = game['away_team']

        if home_team not in team_metrics or away_team not in team_metrics:
            print(f"Metrics not available for one or both teams: {home_team}, {away_team}. Skipping game.")
            continue

        for team_name, opposing_metrics in [(home_team, team_metrics[away_team]), (away_team, team_metrics[home_team])]:
            try:
                team_roster = commonteamroster.CommonTeamRoster(
                    team_id=teams.find_team_by_abbreviation(team_name)['id']
                ).get_data_frames()[0]

                for _, player in team_roster.iterrows():
                    player_id = player['PLAYER_ID']
                    player_name = player['PLAYER']
                    print(f"Fetching data for {player_name} ({team_name})")

                    game_logs = fetch_player_game_logs(player_id)
                    if game_logs.empty:
                        continue

                    mean_ppg, std_dev_ppg, cv_ppg = calculate_stats(game_logs['PTS'])
                    mean_rpg, std_dev_rpg, cv_rpg = calculate_stats(game_logs['REB'])
                    mean_apg, std_dev_apg, cv_apg = calculate_stats(game_logs['AST'])

                    adjusted_ppg = adjust_prediction(mean_ppg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_rpg = adjust_prediction(mean_rpg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)
                    adjusted_apg = adjust_prediction(mean_apg, team_metrics[team_name], opposing_metrics, avg_pace, avg_off_rating, avg_def_rating)

                    all_players_data.append({
                        'Player': player_name,
                        'Team': team_name,
                        'PPG': round(mean_ppg,2),
                        'PPG SD': round(std_dev_ppg, 2),
                        'PPG CV': round(cv_ppg, 2),
                        'RPG': round(mean_rpg, 2),
                        'RPG SD': round(std_dev_rpg, 2),
                        'RPG CV': round(cv_rpg, 2),
                        'APG': round(mean_apg, 2),
                        'APG SD': round(std_dev_apg, 2),
                        'APG CV': round(cv_apg, 2),
                        'Adjusted PPG': round(adjusted_ppg, 2),
                        'Adjusted RPG': round(adjusted_rpg, 2),
                        'Adjusted APG': round(adjusted_apg, 2)
                    })
                    # Add a sleep delay to respect API request limits
                    time.sleep(1)  # Sleep for 1 second between requests
            except Exception as e:
                print(f"Error analyzing team {team_name} or player {player_name}: {e}")
                continue

    # Display the players data
    if all_players_data:
        df = pd.DataFrame(all_players_data)
        display(df)
        return df
    else:
        return pd.DataFrame()

# Start the analysis
analyze_teams_for_today()

from google.colab import data_table
data_table.enable_dataframe_formatter()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.9/284.9 kB 7.9 MB/s eta 0:00:00
Fetching data for Miles Bridges (CHA)
Error analyzing team CHA or player Miles Bridges: name 'fetch_player_game_logs' is not defined
Fetching data for Zyon Pullin (MEM)
Error analyzing team MEM or player Zyon Pullin: name 'fetch_player_game_logs' is not defined
Fetching data for Max Strus (CLE)
Error analyzing team CLE or player Max Strus: name 'fetch_player_game_logs' is not defined
Fetching data for Coby White (CHI)
Error analyzing team CHI or player Coby White: name 'fetch_player_game_logs' is not defined
Fetching data for Tyrese Haliburton (IND)
Error analyzing team IND or player Tyrese Haliburton: name 'fetch_player_game_logs' is not defined
Fetching data for Bilal Coulibaly (WAS)
Error analyzing team WAS or player Bilal Coulibaly: name 'fetch_player_game_logs' is not defined
Fetching data for Anthony Black (ORL)
Error analyzing team ORL or player Anthony Black: name 'fetch_player_game_logs' is not def

KeyboardInterrupt: 